# From Raw Data to Business Decisions
**BITS Pilani Digital | Data Visualization and Storytelling | Assignment 1**

I examine the supplied retail transactions for 2024–2026 to explain sales, profitability,
regional and product differences, and the relationship with discounts. I use Python for a
reproducible preparation process and a Streamlit/Plotly dashboard to communicate the findings.
All results come from the supplied workbooks; the original files remain unchanged.

The student guide defines INR (₹), discounts as fractions, Unit_Cost per item and shipping
expense per transaction. Order_ID is a transaction identifier. In these supplied files, each
retained transaction contains one product, and Order_ID is unique after exact deduplication.
I do not import assumptions from the separate Data Preprocessing assignment.

## Task 1 — Understanding the Business Questions

| Business question | Relevant measures | Required comparisons | Proposed visualization |
|---|---|---|---|
| Q1: How have sales changed, and are patterns seasonal? | Net sales, profit, annual sales growth | Full years; matching calendar months across three years | Annual grouped bars and monthly multi-series line |
| Q2: Which regions lead or lag, and does revenue imply profitability? | Net sales, profit, ratio-of-sums margin, sales share | Region rankings on sales, profit and margin | Grouped horizontal bars with a margin table |
| Q3: Which products contribute most and least? | Category/subcategory sales, profit, contributions and margins | Categories; category-qualified subcategories | Category grouped bars and a ranked subcategory-profit bar chart |
| Q4: How are discounts associated with sales and profitability? | Discount fraction, net sales, profit, margin | Transactions and documented discount bands; category context | Margin-versus-discount scatter and band-margin bars |
| Q5: Which two actions should management prioritize? | Evidence from Q1–Q4 | Size and practicality of opportunities; checks before action | Two evidence-linked recommendations beside the connected story |

Net sales describes revenue after discount, while profit subtracts product cost and shipping.
Considering both avoids treating sales growth as automatically beneficial. Comparable full-year
and calendar-month views help separate isolated peaks from patterns repeated across years.
The discount comparison is descriptive: product mix, quantity and costs can differ across bands.

## Task 2 — Data preparation and integration
### Inspecting the files before combining them
I inspect every supplied sheet, including the student guide. The annual tables should share
the same schema; master tables have a different grain and must have unique keys for safe joins.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from retail_metrics import derive_measures, summarize, annual_summary, monthly_sales
DATA = ROOT / 'Data'
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.3f}'.format)

guide = pd.read_excel(DATA / 'BITS_Retail_Student_Guide.xlsx', sheet_name=None)
for sheet, frame in guide.items():
    print(f'\nStudent guide — {sheet}: field definitions and business conventions')
    display(frame)

annual = {}
source_rows = []
for year in [2024, 2025, 2026]:
    path = DATA / f'BITS_Retail_{year}.xlsx'
    workbook = pd.ExcelFile(path)
    print(f'\n{path.name} — sheets: {workbook.sheet_names}')
    frame = pd.read_excel(path, sheet_name=workbook.sheet_names[0])
    annual[year] = frame
    source_rows.append([year, len(frame), frame.shape[1], int(frame.duplicated().sum())])
    print('Dimensions (rows, columns):', frame.shape, '| columns:', frame.columns.tolist())
    print('Inferred types and non-null counts:')
    frame.info()
    print('First five transaction records:')
    display(frame.head())
    print('Missing values by field:')
    display(frame.isna().sum().rename('Missing').to_frame())
    print('Descriptive statistics for numeric business fields:')
    display(frame[['Quantity','Unit_Price','Discount_Pct','Shipping_Cost']].describe())
source_summary = pd.DataFrame(source_rows, columns=['Source year','Rows','Columns','Exact duplicate copies'])
print('\nAnnual source inventory: raw rows, schema size and exact duplicates')
display(source_summary)

masters = pd.read_excel(DATA / 'BITS_Retail_Master.xlsx', sheet_name=None)
products = masters['Product_Master'].copy()
customers = masters['Customer_Master'].copy()
for sheet, frame in masters.items():
    print(f'\nMaster sheet {sheet}: shape {frame.shape}; columns {frame.columns.tolist()}')
    frame.info()
    print('First five master records:')
    display(frame.head())
    print('Missing values and exact duplicate copies:')
    display(frame.isna().sum().rename('Missing').to_frame())
    print('Exact duplicate copies:', frame.duplicated().sum())
    print('Master-field descriptive summary (keys treated as identifiers):')
    display(frame.describe(include='all'))


Student guide — Data_Guide: field definitions and business conventions


,Field,Definition
0,Dataset,Retail transactions for 2024-2026
1,Currency,INR (₹)
2,Unit_Price,Selling price per item before discount
3,Discount_Pct,Fraction of unit price discounted: 0.10 means 10%
4,Unit_Cost,Cost to retailer for one unit; sourced from Pr...
5,Shipping_Cost,Shipping expense for entire transaction
6,Order_ID,Unique identifier for a transaction
7,Order_Date,Transaction date in YYYY-MM-DD format
8,Region,Region where transaction occurred
9,Product_ID,Key for Product Master



BITS_Retail_2024.xlsx — sheets: ['Orders_2024']
Dimensions (rows, columns): (362, 9) | columns: ['Order_ID', 'Order_Date', 'Customer_ID', 'Product_ID', 'Region', 'Quantity', 'Unit_Price', 'Discount_Pct', 'Shipping_Cost']
Inferred types and non-null counts:
<class 'pandas.DataFrame'>
RangeIndex: 362 entries, 0 to 361
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Order_ID       362 non-null    str    
 1   Order_Date     362 non-null    str    
 2   Customer_ID    362 non-null    str    
 3   Product_ID     362 non-null    str    
 4   Region         362 non-null    str    
 5   Quantity       362 non-null    int64  
 6   Unit_Price     362 non-null    float64
 7   Discount_Pct   362 non-null    float64
 8   Shipping_Cost  362 non-null    float64
dtypes: float64(3), int64(1), str(5)
memory usage: 38.6 KB
First five transaction records:


,Order_ID,Order_Date,Customer_ID,Product_ID,Region,Quantity,Unit_Price,Discount_Pct,Shipping_Cost
0,ORD-2024-0001,2024-05-19,C0211,P013,East,1,407.930,0.050,191.000
1,ORD-2024-0002,2024-05-11,C0135,P006,North,1,"9,259.470",0.030,204.250
2,ORD-2024-0003,2024-11-28,C0175,P011,East,3,328.670,0.130,231.540
3,ORD-2024-0004,2024-03-17,C0193,P011,East,12,353.290,0.000,408.700
4,ORD-2024-0005,2024-09-04,C0066,P013,South,3,428.860,0.000,209.410


Missing values by field:


,Missing
Order_ID,0
Order_Date,0
Customer_ID,0
Product_ID,0
Region,0
Quantity,0
Unit_Price,0
Discount_Pct,0
Shipping_Cost,0


... (26 outputs left out)

Missing values and exact duplicate copies:


,Missing
Customer_ID,0
Customer_Segment,0


Exact duplicate copies: 0
Master-field descriptive summary (keys treated as identifiers):


,Customer_ID,Customer_Segment
count,220,220
unique,220,3
top,C0001,Consumer
freq,1,114


### Appending annual transactions and assessing quality
I preserve every column and record the source year. Repeated customer or product IDs represent
valid repeat activity; exact duplicate transactions are assessed separately. The date check
compares recorded transaction years with source files, without truncating the supplied 2026 data.

In [2]:
transaction_columns = annual[2024].columns.tolist()
assert all(frame.columns.tolist() == transaction_columns for frame in annual.values())
orders = pd.concat([annual[year].assign(Source_Year=year) for year in [2024,2025,2026]], ignore_index=True)
print('Append operation: raw annual schemas agree; no columns discarded.')
print('Rows loaded by year:', {year: len(frame) for year,frame in annual.items()})
print('Rows after pd.concat:', len(orders))
assert len(orders) == sum(len(frame) for frame in annual.values())

orders['Order_Date'] = pd.to_datetime(orders.Order_Date, format='%Y-%m-%d', errors='coerce')
invalid_dates = orders.Order_Date.isna()
wrong_year = orders.Order_Date.dt.year.ne(orders.Source_Year)
duplicate_mask = orders.duplicated(subset=transaction_columns)
missing_required = orders[transaction_columns].isna().any(axis=1)
numeric = ['Quantity','Unit_Price','Discount_Pct','Shipping_Cost']
invalid_numeric = (~np.isfinite(orders[numeric])).any(axis=1) | orders.Quantity.le(0) | orders.Quantity.mod(1).ne(0) | orders.Unit_Price.le(0) | ~orders.Discount_Pct.between(0,1) | orders.Shipping_Cost.lt(0)
print('\nTransaction-quality counts before cleaning (conditions can overlap)')
display(pd.Series({'Invalid dates':invalid_dates.sum(), 'Date/source-year mismatch':wrong_year.sum(),
                   'Rows with missing required fields':missing_required.sum(),
                   'Invalid numeric business values':invalid_numeric.sum(),
                   'Exact duplicate copies':duplicate_mask.sum()},name='Rows').to_frame())
print('Raw region labels:', sorted(orders.Region.unique()))
print('Raw product-category labels:', [repr(x) for x in products.Category.unique()])
print('Customer segments:', sorted(customers.Customer_Segment.unique()))
assert not (invalid_dates | wrong_year | missing_required | invalid_numeric).any()

Append operation: raw annual schemas agree; no columns discarded.
Rows loaded by year: {2024: 362, 2025: 442, 2026: 512}
Rows after pd.concat: 1316

Transaction-quality counts before cleaning (conditions can overlap)


,Rows
Invalid dates,0
Date/source-year mismatch,0
Rows with missing required fields,0
Invalid numeric business values,0
Exact duplicate copies,6


Raw region labels: ['Central', 'East', 'North', 'South', 'West']
Raw product-category labels: ["'Technology'", "'Tech'", "'Furniture'", "'Office Supplies'", "'Office Suply'", "'Office Suppllies'"]
Customer segments: ['Consumer', 'Corporate', 'Small Business']


### Normalizing labels and merging master data
I use a documented mapping for verified aliases, rather than fuzzy matching. Product and
customer keys must be unique in their respective masters. Left joins preserve transactions,
and `many_to_one` prevents duplicated master keys from multiplying their values.

In [3]:
category_mapping = {'Tech':'Technology', 'Office Suply':'Office Supplies', 'Office Suppllies':'Office Supplies'}
before_category = products.Category.copy()
trimmed_category = before_category.str.strip()
products['Category'] = trimmed_category.replace(category_mapping)
category_changes = products.loc[before_category.ne(products.Category), ['Product_ID','Category']].copy()
category_changes.insert(1, 'Original category', before_category[before_category.ne(products.Category)])
print('Category corrections: original master labels and normalized categories')
display(category_changes)
print('Explicit alias mapping:', category_mapping)
assert set(products.Category) == {'Technology','Furniture','Office Supplies'}
assert products.Product_ID.is_unique and customers.Customer_ID.is_unique
assert products.notna().all().all() and customers.notna().all().all()
assert np.isfinite(products.Unit_Cost).all() and products.Unit_Cost.gt(0).all()

# Demonstrate separate merges with indicators and row-count/cardinality checks.
before_product_merge = len(orders)
integrated = orders.merge(products, on='Product_ID', how='left', validate='many_to_one', indicator=True)
product_matches = integrated['_merge'].eq('both').sum()
unmatched_products = integrated.loc[integrated['_merge'].eq('left_only'),'Product_ID'].unique().tolist()
integrated = integrated.drop(columns='_merge')
before_customer_merge = len(integrated)
integrated = integrated.merge(customers, on='Customer_ID', how='left', validate='many_to_one', indicator=True)
customer_matches = integrated['_merge'].eq('both').sum()
unmatched_customers = integrated.loc[integrated['_merge'].eq('left_only'),'Customer_ID'].unique().tolist()
integrated = integrated.drop(columns='_merge')
merge_audit = pd.DataFrame([
    ['Product master', before_product_merge, before_customer_merge, product_matches, len(unmatched_products), products.Product_ID.duplicated().sum()],
    ['Customer master', before_customer_merge, len(integrated), customer_matches, len(unmatched_customers), customers.Customer_ID.duplicated().sum()]],
    columns=['Merge','Rows before','Rows after','Matched rows','Unmatched keys','Duplicate master keys'])
print('\nMaster-join audit: both joins must preserve transaction counts')
display(merge_audit)
print('Unmatched product keys:',unmatched_products,'| unmatched customer keys:',unmatched_customers)
assert len(integrated) == len(orders) and not unmatched_products and not unmatched_customers

Category corrections: original master labels and normalized categories


,Product_ID,Original category,Category
2,P003,Tech,Technology
11,P012,Office Suply,Office Supplies
13,P014,Office Suppllies,Office Supplies


Explicit alias mapping: {'Tech': 'Technology', 'Office Suply': 'Office Supplies', 'Office Suppllies': 'Office Supplies'}



Master-join audit: both joins must preserve transaction counts


,Merge,Rows before,Rows after,Matched rows,Unmatched keys,Duplicate master keys
0,Product master,1316,1316,1316,0,0
1,Customer master,1316,1316,1316,0,0


Unmatched product keys: [] | unmatched customer keys: []


### Cleaning policy and guide-based measures
I remove extra fully identical transaction copies, retaining the first. The guide defines
Order_ID as a unique transaction identifier; I verify that no conflicting repeated IDs remain.
Positive unusual quantities, prices and shipping expenses are retained when valid. I do not
remove transactions simply because they produce losses.

The financial definitions use the supplied guide's raw-field meanings:

| Measure | Calculation |
|---|---|
| GrossSales | Quantity × Unit_Price |
| DiscountAmount | GrossSales × Discount_Pct (a fraction) |
| NetSales | GrossSales − DiscountAmount |
| ProductCost | Quantity × Unit_Cost |
| Profit | NetSales − ProductCost − Shipping_Cost, with shipping subtracted once per transaction |
| ProfitMarginPct | 100 × Profit / NetSales |

For group margins I divide summed profit by summed net sales. A gross-sales-weighted discount
is summed DiscountAmount divided by summed GrossSales; it is not the simple transaction average.
The guide supplies field conventions rather than a prewritten profit equation, so this expense
definition is stated explicitly. No taxes, returns or other unavailable costs are invented.

In [4]:
cleaned = integrated.drop_duplicates(subset=transaction_columns).copy()
assert cleaned.Order_ID.is_unique, 'Investigate conflicting non-identical records sharing Order_ID'
cleaned = derive_measures(cleaned)
print('Cleaning rows: appended / after exact deduplication:',len(integrated),'/',len(cleaned))
print('Financial preview: raw inputs and each derived monetary component (INR)')
display(cleaned[['Order_ID','Quantity','Unit_Price','Discount_Pct','Unit_Cost','Shipping_Cost',
                 'GrossSales','DiscountAmount','NetSales','ProductCost','Profit','ProfitMarginPct']].head())

outlier_rows = []
for field in ['Quantity','Unit_Price','Shipping_Cost']:
    q1,q3 = cleaned[field].quantile([.25,.75]); iqr=q3-q1
    lower,upper = q1-1.5*iqr,q3+1.5*iqr
    flagged = ((cleaned[field]<lower)|(cleaned[field]>upper)).sum()
    outlier_rows.append([field,lower,upper,int(flagged),'Retained; valid positive values, not proven errors'])
outliers = pd.DataFrame(outlier_rows,columns=['Field','Lower IQR fence','Upper IQR fence','Flagged rows','Decision'])
print('\nUnusual-value assessment on retained transactions; these are flags, not errors')
display(outliers)
print('Product-price context: price ranges differ by category')
display(cleaned.groupby('Category').Unit_Price.agg(['min','median','max']))
print('Loss-making transactions retained:',int(cleaned.Profit.lt(0).sum()))
print('Date coverage:',cleaned.Order_Date.min().date(),'through',cleaned.Order_Date.max().date())
print('Transactions after 15 November 2026:',int(cleaned.Order_Date.gt('2026-11-15').sum()))

Cleaning rows: appended / after exact deduplication:

 1316 / 1310
Financial preview: raw inputs and each derived monetary component (INR)


,Order_ID,Quantity,Unit_Price,Discount_Pct,Unit_Cost,Shipping_Cost,GrossSales,DiscountAmount,NetSales,ProductCost,Profit,ProfitMarginPct
0,ORD-2024-0001,1,407.930,0.050,250,191.000,407.930,20.397,387.534,250,-53.466,-13.797
1,ORD-2024-0002,1,"9,259.470",0.030,6700,204.250,"9,259.470",277.784,"8,981.686",6700,"2,077.436",23.130
2,ORD-2024-0003,3,328.670,0.130,230,231.540,986.010,128.181,857.829,690,-63.711,-7.427
3,ORD-2024-0004,12,353.290,0.000,230,408.700,"4,239.480",0.000,"4,239.480",2760,"1,070.780",25.257
4,ORD-2024-0005,3,428.860,0.000,250,209.410,"1,286.580",0.000,"1,286.580",750,327.170,25.429



Unusual-value assessment on retained transactions; these are flags, not errors


,Field,Lower IQR fence,Upper IQR fence,Flagged rows,Decision
0,Quantity,-2.000,6.000,67,"Retained; valid positive values, not proven er..."
1,Unit_Price,"-17,521.147","31,408.012",103,"Retained; valid positive values, not proven er..."
2,Shipping_Cost,-3.734,468.296,83,"Retained; valid positive values, not proven er..."


Product-price context: price ranges differ by category


,min,median,max
Category,,,
Furniture,"1,782.910","7,377.975","14,533.660"
Office Supplies,259.290,426.080,"2,046.160"
Technology,816.620,"12,305.710","56,731.080"


Loss-making transactions retained: 108
Date coverage: 2024-01-01 through 2026-12-28
Transactions after 15 November 2026: 84


The supplied academic dataset covers the complete 2024–2026 scope, including dates after
15 November 2026. I retain these supplied records rather than truncating 2026 or claiming that
they represent verified real-world sales already completed. The IQR table documents unusually
large valid purchases; product-price differences provide a reason not to cap prices globally.

In [5]:
cleaning_log = pd.DataFrame([
    ['Exact duplicate copies','Annual transactions',int(duplicate_mask.sum()),'Removed extra copies; retained first','Guide identifies Order_ID as a unique transaction','Order_ID unique after cleaning'],
    ['Category spelling/aliases','Product_Master.Category',int(trimmed_category.ne(products.Category).sum()),'Mapped Tech, Office Suply and Office Suppllies to documented categories','Genuine aliases of supplied business categories','Exactly three normalized categories'],
    ['Outer category whitespace','Product_Master.Category',int(before_category.ne(trimmed_category).sum()),'Trimmed outer whitespace','Formatting does not define a new category','No outer category whitespace remains'],
    ['Missing required transaction values','Transaction fields',int(missing_required.sum()),'No records removed','No missing inputs to required measures','All required fields present'],
    ['Unmatched master keys','Product_ID / Customer_ID',len(unmatched_products)+len(unmatched_customers),'Retained all matched transactions; no attributes invented','No unmatched keys found','Both joins preserve row counts'],
    ['Invalid dates or source-year mismatch','Order_Date',int((invalid_dates|wrong_year).sum()),'Parsed dates; no records removed','Scope must agree with source year','All dates valid and agree with source year'],
    ['Invalid numeric values','Quantity, prices, discount, shipping',int(invalid_numeric.sum()),'No correction or removal needed','Positive quantities/prices; discounts 0-1; nonnegative shipping','All numeric validity assertions pass'],
    *[['IQR statistical extremes',row['Field'],row['Flagged rows'],'Retained without capping','Unusual but valid values are not necessarily errors','No extra rows removed'] for row in outliers.to_dict('records')],
    ['Negative calculated profit','Profit',int(cleaned.Profit.lt(0).sum()),'Retained losses','A valid transaction may lose money','Losses remain visible in all summaries']
],columns=['Issue','Field','Records affected','Action taken','Justification','Validation outcome'])
print('Cleaning log: records removed, labels corrected and valid unusual records retained')
display(cleaning_log)

# Independent arithmetic checks on representative first, middle and last records.
for row in cleaned.iloc[[0,len(cleaned)//2,-1]].itertuples():
    expected_net = row.Quantity*row.Unit_Price*(1-row.Discount_Pct)
    expected_profit = expected_net-row.Quantity*row.Unit_Cost-row.Shipping_Cost
    assert np.isclose(row.NetSales,expected_net) and np.isclose(row.Profit,expected_profit)
assert np.isfinite(cleaned[['NetSales','ProductCost','Profit','ProfitMarginPct']]).all().all()
assert cleaned.DiscountBand.notna().all() and cleaned.Order_ID.is_unique
assert (cleaned.Year==cleaned.Source_Year).all()
for field in ['Year','Region','Category']:
    assert np.isclose(cleaned.groupby(field).NetSales.sum().sum(),cleaned.NetSales.sum())
    assert np.isclose(cleaned.groupby(field).Profit.sum().sum(),cleaned.Profit.sum())

cleaned.to_csv(ROOT / 'output/BITS_Retail_Cleaned_Integrated.csv',index=False,date_format='%Y-%m-%d')
cleaning_log.to_csv(ROOT / 'output/Data_Cleaning_Log.csv',index=False)
reloaded = pd.read_csv(ROOT / 'output/BITS_Retail_Cleaned_Integrated.csv')
assert len(reloaded)==len(cleaned) and reloaded.Order_ID.is_unique
assert np.allclose(reloaded.NetSales,cleaned.NetSales) and np.allclose(reloaded.Profit,cleaned.Profit)
print('All preparation checks passed. Exported transaction-level rows:',len(reloaded))
print('The cleaned CSV preserves original transaction inputs, master attributes and derived measures.')

Cleaning log: records removed, labels corrected and valid unusual records retained


,Issue,Field,Records affected,Action taken,Justification,Validation outcome
0,Exact duplicate copies,Annual transactions,6,Removed extra copies; retained first,Guide identifies Order_ID as a unique transaction,Order_ID unique after cleaning
1,Category spelling/aliases,Product_Master.Category,3,"Mapped Tech, Office Suply and Office Suppllies...",Genuine aliases of supplied business categories,Exactly three normalized categories
2,Outer category whitespace,Product_Master.Category,0,Trimmed outer whitespace,Formatting does not define a new category,No outer category whitespace remains
3,Missing required transaction values,Transaction fields,0,No records removed,No missing inputs to required measures,All required fields present
4,Unmatched master keys,Product_ID / Customer_ID,0,Retained all matched transactions; no attribut...,No unmatched keys found,Both joins preserve row counts
5,Invalid dates or source-year mismatch,Order_Date,0,Parsed dates; no records removed,Scope must agree with source year,All dates valid and agree with source year
6,Invalid numeric values,"Quantity, prices, discount, shipping",0,No correction or removal needed,Positive quantities/prices; discounts 0-1; non...,All numeric validity assertions pass
7,IQR statistical extremes,Quantity,67,Retained without capping,Unusual but valid values are not necessarily e...,No extra rows removed
8,IQR statistical extremes,Unit_Price,103,Retained without capping,Unusual but valid values are not necessarily e...,No extra rows removed
9,IQR statistical extremes,Shipping_Cost,83,Retained without capping,Unusual but valid values are not necessarily e...,No extra rows removed


All preparation checks passed. Exported transaction-level rows: 1310
The cleaned CSV preserves original transaction inputs, master attributes and derived measures.


## Task 3 — Business analysis and visualization selection
### Q1: Sales and profit across years and months
I compare full-year totals and consecutive-year growth, then align monthly net sales on the
same January–December axis. Annual bars show scale; monthly lines help assess whether peaks
repeat. A single high month is insufficient evidence of seasonality.

In [6]:
annual_results = annual_summary(cleaned)
monthly_results = monthly_sales(cleaned)
print('Annual performance: INR totals, ratio-of-sums margin and consecutive-year sales growth')
display(annual_results)
print('Monthly net sales: comparable calendar months, columns are years (INR)')
display(monthly_results.pivot(index='Month',columns='Year',values='NetSales'))
print('Peak net-sales month in each supplied year')
display(monthly_results.loc[monthly_results.groupby('Year').NetSales.idxmax()])
print('Fourth-quarter net-sales share within each year (%)')
display((monthly_results[monthly_results.Month.ge(10)].groupby('Year').NetSales.sum()/cleaned.groupby('Year').NetSales.sum()*100).rename('Q4 sales share').to_frame())

Annual performance: INR totals, ratio-of-sums margin and consecutive-year sales growth


,Year,NetSales,Profit,GrossSales,DiscountAmount,Orders,Transactions,ProfitMarginPct,GrossWeightedDiscountPct,SalesSharePct,ProfitSharePct,YoYSalesPct
0,2024,"7,792,246.658","946,010.828","8,688,907.770","896,661.112",360,360,12.140,10.320,22.128,16.304,NaN
1,2025,"11,613,237.356","1,838,594.586","12,788,017.150","1,174,779.794",440,440,15.832,9.187,32.978,31.686,49.036
2,2026,"15,809,532.806","3,017,865.706","17,254,134.880","1,444,602.074",510,510,19.089,8.372,44.894,52.010,36.134


Monthly net sales: comparable calendar months, columns are years (INR)


Year,2024,2025,2026
Month,,,
1,"375,985.817","492,853.949","912,237.006"
2,"211,421.081","541,853.969","1,154,039.660"
3,"317,439.305","739,367.097","559,601.330"
4,"491,461.064","878,510.546","1,540,432.359"
5,"752,424.441","937,519.938","972,686.834"
6,"516,540.700","1,469,899.354","1,147,942.868"
7,"1,057,194.562","1,011,021.720","1,109,977.315"
8,"376,740.921","616,218.938","1,318,747.974"
9,"956,187.685","1,219,008.518","1,890,282.410"


Peak net-sales month in each supplied year


,Year,Month,NetSales
9,2024,10,"1,178,887.942"
17,2025,6,"1,469,899.354"
33,2026,10,"2,310,176.814"


Fourth-quarter net-sales share within each year (%)


,Q4 sales share
Year,
2024,35.123
2025,31.920
2026,32.914


### Q2: Regional sales and profitability
I compare net sales, absolute profit and the ratio-of-sums margin. Grouped horizontal bars
support ranking on a common INR scale, while the accompanying table makes margins explicit.
Contributions always use the total within the same filter scope.

In [7]:
regional_results = summarize(cleaned,'Region').sort_values('NetSales',ascending=False)
print('Regional performance ranked by net sales: profit, margin and contribution remain separate measures')
display(regional_results)
print('Regional performance by year: does a pooled ranking hide a change?')
display(summarize(cleaned,['Year','Region'])[['Year','Region','NetSales','Profit','ProfitMarginPct']])

Regional performance ranked by net sales: profit, margin and contribution remain separate measures


,Region,NetSales,Profit,GrossSales,DiscountAmount,Orders,Transactions,ProfitMarginPct,GrossWeightedDiscountPct,SalesSharePct,ProfitSharePct
4,West,"10,564,589.488","1,770,597.658","11,598,158.060","1,033,568.572",350,350,16.760,8.911,30.000,30.515
2,North,"9,544,035.723","1,653,760.532","10,351,490.270","807,454.547",314,314,17.328,7.800,27.102,28.501
1,East,"6,965,793.732","1,158,264.332","7,621,278.560","655,484.828",254,254,16.628,8.601,19.781,19.962
3,South,"4,261,887.798","704,676.108","4,741,476.120","479,588.322",245,245,16.534,10.115,12.102,12.144
0,Central,"3,878,710.081","515,172.491","4,418,656.790","539,946.709",147,147,13.282,12.220,11.014,8.879


Regional performance by year: does a pooled ranking hide a change?


,Year,Region,NetSales,Profit,ProfitMarginPct
0,2024,Central,"1,160,063.727","121,607.847",10.483
1,2024,East,"1,251,422.772","194,667.392",15.556
2,2024,North,"1,935,258.827","260,769.167",13.475
3,2024,South,"1,135,347.843","146,038.973",12.863
4,2024,West,"2,310,153.489","222,927.449",9.650
5,2025,Central,"1,648,279.923","208,041.443",12.622
6,2025,East,"1,872,266.326","294,661.746",15.738
7,2025,North,"2,671,272.288","475,792.618",17.811
8,2025,South,"1,619,831.112","246,307.452",15.206
9,2025,West,"3,801,587.706","613,791.326",16.146


### Q3: Category and subcategory contribution
I use normalized categories and consider sales alongside profit. A category-qualified subcategory
label keeps Furniture/Storage distinct from Office Supplies/Storage. The dashboard ranks
subcategory profit with a visible zero line, so losses cannot be hidden by positive revenue.

In [8]:
category_results = summarize(cleaned,'Category').sort_values('NetSales',ascending=False)
subcategory_results = summarize(cleaned,['Category','Subcategory']).sort_values('Profit',ascending=False)
print('Category contribution: net sales, profit, margin, and shares of the matching totals')
display(category_results)
print('Category/subcategory performance ranked by profit')
display(subcategory_results)

Category contribution: net sales, profit, margin, and shares of the matching totals


,Category,NetSales,Profit,GrossSales,DiscountAmount,Orders,Transactions,ProfitMarginPct,GrossWeightedDiscountPct,SalesSharePct,ProfitSharePct
2,Technology,"27,565,779.292","4,342,748.032","30,104,294.210","2,538,514.918",565,565,15.754,8.432,78.278,74.843
0,Furniture,"6,961,185.440","1,308,307.020","7,870,376.900","909,191.460",340,340,18.794,11.552,19.768,22.547
1,Office Supplies,"688,052.088","151,416.068","756,388.690","68,336.602",405,405,22.006,9.035,1.954,2.610


Category/subcategory performance ranked by profit


,Category,Subcategory,NetSales,Profit,GrossSales,DiscountAmount,Orders,Transactions,ProfitMarginPct,GrossWeightedDiscountPct,SalesSharePct,ProfitSharePct
9,Technology,Computers,"13,989,541.005","1,923,425.595","15,194,661.470","1,205,120.465",102,102,13.749,7.931,39.726,33.148
11,Technology,Phones,"8,920,505.622","1,251,193.132","9,816,115.210","895,609.588",131,131,14.026,9.124,25.332,21.563
10,Technology,Monitors,"3,107,472.647","730,087.607","3,409,506.110","302,033.463",110,110,23.495,8.859,8.824,12.582
2,Furniture,Desks,"3,082,579.416","492,142.486","3,494,940.720","412,361.304",90,90,15.965,11.799,8.754,8.482
3,Furniture,Storage,"2,420,503.655","482,861.295","2,748,537.430","328,033.775",133,133,19.949,11.935,6.873,8.322
8,Technology,Accessories,"1,548,260.019","438,041.699","1,684,011.420","135,751.401",222,222,28.293,8.061,4.397,7.549
1,Furniture,Chairs,"1,200,913.616","256,440.956","1,337,590.980","136,677.364",53,53,21.354,10.218,3.410,4.420
0,Furniture,Accessories,"257,188.754","76,862.284","289,307.770","32,119.016",64,64,29.886,11.102,0.730,1.325
4,Office Supplies,Ink,"295,979.375","71,811.405","327,444.570","31,465.195",73,73,24.262,9.609,0.840,1.238
6,Office Supplies,Storage,"183,592.926","51,763.816","201,347.290","17,754.364",88,88,28.195,8.818,0.521,0.892


### Q4: Discounting, sales and profitability
I compare transaction margins against discount rates and aggregate discount bands. Marker area
shows net sales in the scatter plot; category color supplies context. The band chart summarizes
margin as total profit / total net sales. Boundaries are 0%, (0,10%], (10%,20%], and >20%.
Rounding the grouping key avoids floating-point artifacts at boundaries; monetary calculations
retain the supplied discount fractions. These comparisons describe associations, not causal effects.

In [9]:
discount_results = summarize(cleaned,'DiscountBand')
discount_by_category = summarize(cleaned,['Category','DiscountBand'])
print('Discount-band performance: boundaries are documented above; all transactions are included')
display(discount_results)
print('Discount-band margins within categories: product-mix context')
display(discount_by_category[['Category','DiscountBand','Transactions','NetSales','Profit','ProfitMarginPct']])
print('Unweighted mean transaction discount (%):',round(cleaned.Discount_Pct.mean()*100,3))
print('Gross-sales-weighted discount (%):',round(cleaned.DiscountAmount.sum()/cleaned.GrossSales.sum()*100,3))
print('Descriptive Pearson correlation: discount fraction vs transaction profit margin:',round(cleaned.Discount_Pct.corr(cleaned.ProfitMarginPct),3))

Discount-band performance: boundaries are documented above; all transactions are included


,DiscountBand,NetSales,Profit,GrossSales,DiscountAmount,Orders,Transactions,ProfitMarginPct,GrossWeightedDiscountPct,SalesSharePct,ProfitSharePct
0,0%,"7,026,521.880","1,672,558.950","7,026,521.880",0.000,232,232,23.804,0.000,19.953,28.825
1,>0-10%,"16,994,373.318","3,047,865.948","18,263,557.200","1,269,183.882",609,609,17.935,6.949,48.259,52.527
2,>10-20%,"10,143,334.042","1,089,468.262","12,021,820.170","1,878,486.128",405,405,10.741,15.626,28.804,18.776
3,>20%,"1,050,787.580","-7,422.040","1,419,160.550","368,372.970",64,64,-0.706,25.957,2.984,-0.128


Discount-band margins within categories: product-mix context


,Category,DiscountBand,Transactions,NetSales,Profit,ProfitMarginPct
0,Furniture,0%,41,"894,873.950","249,069.880",27.833
1,Furniture,>0-10%,142,"3,025,889.800","682,722.620",22.563
2,Furniture,>10-20%,127,"2,483,420.271","367,392.821",14.794
3,Furniture,>20%,30,"557,001.419","9,121.699",1.638
4,Office Supplies,0%,74,"156,496.430","45,499.350",29.074
5,Office Supplies,>0-10%,201,"325,225.851","76,833.971",23.625
6,Office Supplies,>10-20%,113,"179,330.225","27,383.955",15.270
7,Office Supplies,>20%,17,"26,999.582","1,698.792",6.292
8,Technology,0%,117,"5,975,151.500","1,377,989.720",23.062
9,Technology,>0-10%,266,"13,643,257.667","2,288,309.357",16.772


Unweighted mean transaction discount (%): 9.362
Gross-sales-weighted discount (%): 9.078
Descriptive Pearson correlation: discount fraction vs transaction profit margin: -0.494


## Task 4 — Visualization design
The dashboard uses blue for net sales, teal for profit and red for negative profit. Ordinary bars
include zero; margins are percentages on separate axes rather than sharing an INR axis. Years
have consistent colors and line styles. Sorted categories and clear units support comparison;
hover text provides exact values and context without crowding labels. I use four coordinated
filter controls (year, region, category and segment), a reset button and an explicit empty-state
message. Insights tied to current filters are recalculated; management priorities derived from
the full dataset are labeled as such. There are seven visuals, each serving Q1–Q4 rather than
adding decorative charts. Q5 is answered through two actions supported by the resulting story.

## Task 5 ? Connected business insights
Sales increased from ?7.79m in 2024 to ?11.61m in 2025 and ?15.81m in 2026, while profit margin rose from 12.14% to 19.09%. Growth therefore coincides with improving profitability, rather than simply more revenue. Sales growth is 49.04% and 36.13% in consecutive years. October peaks in 2024 and 2026, but June leads in 2025; three years do not establish a reliable recurring seasonal mechanism.

The dashboard then shows where performance is concentrated. West leads both sales (?10.56m) and profit (?1.77m), whereas Central is lowest on both (?3.88m and ?0.52m). North has the strongest regional margin, 17.33%, compared with West at 16.76% and Central at 13.28%. Technology supplies 78.28% of sales and 74.84% of profit, but Office Supplies has the highest category margin (22.01%) on only 1.95% of sales. Computers lead both subcategory sales and profit; Office Supplies/Writing is lowest on both. Revenue scale and margin answer different management questions.

Finally, the discount scatter and band chart identify a targeted profitability risk: 64 transactions discounted above 20% generate ?1.05m in net sales but lose ?7,422.04 overall (?0.71% margin). The zero-discount band earns 23.80%. This association supports investigating approval rules, not a claim that removing discounts would automatically raise profit: category mix, quantity, prices and shipping differ. Together these findings motivate controlled discount safeguards and a focused product-mix test.

## Task 6 ? Two management recommendations
1. **Introduce approval and contribution-margin checks for discounts above 20%.** The 64 transactions in this band lose ?7,422.04 overall. Review product cost and transaction shipping before approval, and test the policy on comparable product/region groups. The intended benefit is fewer loss-making promotions, not a guaranteed financial uplift. Check demand response and volume effects before wider adoption.
2. **Protect Computers availability and pilot Technology/Accessories attachment.** Computers account for 39.73% of all sales at 13.75% margin; Technology/Accessories earns 28.29%. Maintain availability of the major revenue contributor while testing whether relevant accessories can improve contribution. Verify stock constraints, customer demand and incremental margin in a small controlled bundle pilot. The available data does not prove existing cross-selling demand or quantify a future uplift.